Published on May 12, 2023. By Marília Prata, mpwolke.

In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)
import seaborn as sns
import matplotlib.pyplot as plt

import plotly.express as px
import plotly.graph_objs as go

import plotly
plotly.offline.init_notebook_mode(connected=True)

#Ignore warnings
import warnings
warnings.filterwarnings('ignore')

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/opt/conda/lib/python3.10/site-packages/scipy/__init__.py:146: UserWarning: A NumPy version >=1.16.5 and <1.23.0 is required for this version of SciPy (detected version 1.23.5
  warnings.warn(f"A NumPy version >={np_minversion} and <{np_maxversion}"


/kaggle/input/icr-identify-age-related-conditions/sample_submission.csv
/kaggle/input/icr-identify-age-related-conditions/greeks.csv
/kaggle/input/icr-identify-age-related-conditions/train.csv
/kaggle/input/icr-identify-age-related-conditions/test.csv


#Competition Citation

"The competition data comprises over fifty anonymized health characteristics linked to three age-related conditions. Your goal is to predict whether a subject has or has not been diagnosed with one of these conditions -- a binary classification problem."
https://www.kaggle.com/competitions/icr-identify-age-related-conditions/data

@misc{icr-identify-age-related-conditions,

    author = {Aaron Carman, Alexander Heifler, Ashley Chow, HCL-Rantig, Ryan Holbrook},
    
    title = {ICR - Identifying Age-Related Conditions},
    
    publisher = {Kaggle},
    
    year = {2023},
    
    url = {https://kaggle.com/competitions/icr-identify-age-related-conditions}

![](https://i.ytimg.com/vi/QFd092Mhq-w/maxresdefault.jpg)youtube.com

#Log Loss

By Dan Becker

Log Loss is the most important classification metric based on probabilities.

It's hard to interpret raw log-loss values, but log-loss is still a good metric for comparing models. For any given problem, a lower log-loss value means better predictions.

Log Loss is a slight twist on something called the Likelihood Function. In fact, Log Loss is -1 * the log of the likelihood function. So, we will start by understanding the likelihood function.

The likelihood function answers the question "How likely did the model think the actually observed set of outcomes was." If that sounds confusing, an example should help.

From Likelihood to Log Loss

"Each prediction is between 0 and 1. If you multiply enough numbers in this range, the result gets so small that computers can't keep track of it. So, as a clever computational trick, we instead keep track of the log of the Likelihood. This is in a range that's easy to keep track of. We multiply this by negative 1 to maintain a common convention that lower loss scores are better."

https://www.kaggle.com/code/dansbecker/what-is-log-loss

In [2]:
train = pd.read_csv("/kaggle/input/icr-identify-age-related-conditions/train.csv", delimiter=',', encoding='UTF-8')
#pd.set_option('display.max_columns', None)
train.head()

,Id,AB,AF,AH,AM,AR,AX,AY,AZ,BC,...,FL,FR,FS,GB,GE,GF,GH,GI,GL,Class
0,000ff2bfdfe9,0.209377,3109.03329,85.200147,22.394407,8.138688,0.699861,0.025578,9.812214,5.555634,...,7.298162,1.73855,0.094822,11.339138,72.611063,2003.810319,22.136229,69.834944,0.120343,1
1,007255e47698,0.145282,978.76416,85.200147,36.968889,8.138688,3.632190,0.025578,13.517790,1.229900,...,0.173229,0.49706,0.568932,9.292698,72.611063,27981.562750,29.135430,32.131996,21.978000,0
2,013f2bd269f5,0.470030,2635.10654,85.200147,32.360553,8.138688,6.732840,0.025578,12.824570,1.229900,...,7.709560,0.97556,1.198821,37.077772,88.609437,13676.957810,28.022851,35.192676,0.196941,0
3,043ac50845d5,0.252107,3819.65177,120.201618,77.112203,8.138688,3.685344,0.025578,11.053708,1.229900,...,6.122162,0.49706,0.284466,18.529584,82.416803,2094.262452,39.948656,90.493248,0.155829,0
4,044fb8a146ec,0.380297,3733.04844,85.200147,14.103738,8.138688,3.942255,0.054810,3.396778,102.151980,...,8.153058,48.50134,0.121914,16.408728,146.109943,8524.370502,45.381316,36.262628,0.096614,1


#Cancer

"Cancer treatments have advanced dramatically in recent years. But ICR researching platforms aimed at detecting cancer at its earliest stages and preventing it altogether."

![](https://img1.wsimg.com/isteam/ip/f276cfbb-1380-4489-b98d-8ff636c6cb81/nci-dividing%20breast%20cancer%20cell.jpg/:/cr=t:25%25,l:0%25,w:100%25,h:50%25/rs=w:388,h:194,cg:true)https://invitrocellresearch.com/our-research

In [3]:
test = pd.read_csv("/kaggle/input/icr-identify-age-related-conditions/test.csv", delimiter=',', encoding='UTF-8')
#pd.set_option('display.max_columns', None)
test.head()

,Id,AB,AF,AH,AM,AR,AX,AY,AZ,BC,...,FI,FL,FR,FS,GB,GE,GF,GH,GI,GL
0,00eed32682bb,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,010ebe33f668,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,02fa521e1838,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,040e15f562a2,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,046e85c7cc7f,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


#Connective Tissue Disease and the Aging ECM

"Aging connective tissue and decaying ECM remain underappreciated and under-researched areas of human aging research and therapeutic development. At ICR, we take several approaches to these issues, including researching connective tissue diseases like Ehlers-Danlos Syndrome and Hypermobility Spectrum Disorder for clues to develop effective therapeutics."

![](https://img1.wsimg.com/isteam/ip/f276cfbb-1380-4489-b98d-8ff636c6cb81/Connective%20Tissue%202.jpeg/:/cr=t:16.67%25,l:0%25,w:100%25,h:66.67%25/rs=w:388,h:194,cg:true)https://invitrocellresearch.com/our-research

In [4]:
sub = pd.read_csv("/kaggle/input/icr-identify-age-related-conditions/sample_submission.csv", delimiter=',', encoding='UTF-8')
#pd.set_option('display.max_columns', None)
sub.head()

,Id,class_0,class_1
0,00eed32682bb,0.5,0.5
1,010ebe33f668,0.5,0.5
2,02fa521e1838,0.5,0.5
3,040e15f562a2,0.5,0.5
4,046e85c7cc7f,0.5,0.5


#Dementia

"Preventing age-related cognitive decline and dementia goes hand-in-hand with research to repair and restore damaged and lost brain cells at ICR."

![](https://img1.wsimg.com/isteam/ip/f276cfbb-1380-4489-b98d-8ff636c6cb81/Nerve%20repair%20ad%203.jpg/:/rs=w:388,h:194,cg:true,m/cr=w:388,h:194)https://invitrocellresearch.com/our-research

#Omega

"Omega (uppercase Ω, lowercase ω) is the 24th and last letter of the Greek alphabet. In the Greek numeric system, it has a value of 800. Pronounced or 'aw' as in 'raw'."

https://www.tau.ac.il/~tsirel/dump/Static/knowino.org/wiki/Omega_(Greek_letter).html#:~:text=Omega%20(uppercase%20%CE%A9%2C%20lowercase%20%CF%89,'%20as%20in%20'raw'.

#The Greeks csv file

Alpha Identifies the type of age-related condition, if present.

A No age-related condition. Corresponds to class 0.

B, D, G The three age-related conditions. Correspond to class 1.

Beta, Gamma, Delta Three experimental characteristics.

Epsilon The date the data for this subject was collected. Note that all of the data in the test set was collected after the training set was collected.

https://www.kaggle.com/competitions/icr-identify-age-related-conditions/data

In [5]:
greeks = pd.read_csv("/kaggle/input/icr-identify-age-related-conditions/greeks.csv", delimiter=',', encoding='UTF-8')
pd.set_option('display.max_columns', None)
greeks.head()

,Id,Alpha,Beta,Gamma,Delta,Epsilon
0,000ff2bfdfe9,B,C,G,D,3/19/2019
1,007255e47698,A,C,M,B,Unknown
2,013f2bd269f5,A,C,M,B,Unknown
3,043ac50845d5,A,C,M,B,Unknown
4,044fb8a146ec,D,B,F,B,3/25/2020


#Ischemic heart disease

"Ischemic heart disease is the world's biggest killer. ICR professionals are researching practical ways to prevent and reverse atherosclerosis as well as repair aged, damaged vasculature."

![](https://img1.wsimg.com/isteam/ip/f276cfbb-1380-4489-b98d-8ff636c6cb81/Athero.jpg/:/rs=w:388,h:194,cg:true,m/cr=w:388,h:194)https://invitrocellresearch.com/our-research

#Missing Values

Only BQ, CB, CC, DU, EL, FL, FC, FS, GL have missing values.

In [6]:
train.isnull().sum()

Id        0
AB        0
AF        0
AH        0
AM        0
AR        0
AX        0
AY        0
AZ        0
BC        0
BD        0
BN        0
BP        0
BQ       60
BR        0
BZ        0
CB        2
CC        3
CD        0
CF        0
CH        0
CL        0
CR        0
CS        0
CU        0
CW        0
DA        0
DE        0
DF        0
DH        0
DI        0
DL        0
DN        0
DU        1
DV        0
DY        0
EB        0
EE        0
EG        0
EH        0
EJ        0
EL       60
EP        0
EU        0
FC        1
FD        0
FE        0
FI        0
FL        1
FR        0
FS        2
GB        0
GE        0
GF        0
GH        0
GI        0
GL        1
Class     0
dtype: int64

#Stroke

"Stroke is a major cause of death and debilitation in aging adults. Research into stroke prevention and repairing the aging neurovascular system is central to our mission."

![](https://img1.wsimg.com/isteam/ip/f276cfbb-1380-4489-b98d-8ff636c6cb81/stroke1.jpg/:/cr=t:12.07%25,l:0%25,w:100%25,h:45.2%25/rs=w:388,h:194,cg:true)https://invitrocellresearch.com/our-research

In [17]:
from subprocess import check_output
print(check_output(["ls", "../input"]).decode("utf8"))

icr-identify-age-related-conditions



#Osteoarthritis

"Osteoarthritis is a primary contributor to age-related frailty. We are researching novel, practical ways to regenerate cartilage and repair osteoarthritic joints."

![](https://img1.wsimg.com/isteam/ip/f276cfbb-1380-4489-b98d-8ff636c6cb81/OA.jpg/:/cr=t:62.93%25,l:9.68%25,w:80.65%25,h:30.26%25/rs=w:388,h:194,cg:true,m)https://invitrocellresearch.com/our-research

In [18]:
from sklearn.datasets import make_blobs
from sklearn.ensemble import RandomForestClassifier
from sklearn.calibration import CalibratedClassifierCV
from sklearn.metrics import log_loss

np.random.seed(0)

#Atherosclerosis, Neurodegeneration and Frailty


"The aging immune system underlies most age-related disease, including atherosclerosis, cancer, neurodegeneration and frailty. Regenerating the aged immune system is key to successfully combating aging."

![](https://img1.wsimg.com/isteam/ip/f276cfbb-1380-4489-b98d-8ff636c6cb81/Immunoregeneration.jpeg/:/rs=w:388,h:194,cg:true,m/cr=w:388,h:194)https://invitrocellresearch.com/our-research

In [20]:
# Lets first handle numerical features with nan value
numerical_nan = [feature for feature in train.columns if train[feature].isna().sum()>1 and train[feature].dtypes!='O']
numerical_nan

[]

#I think I deleted that snippet after EJ becomes "integer" So on train.dtypes all is float/integer. 

In [ ]:
#Code by Tejashvi14 https://www.kaggle.com/tejashvi14/casualty-analysis/notebook

#After that snippet the float(Year) and the other objects become integers.

train["EJ"]=train["EJ"].apply(int)

In [21]:
from sklearn.preprocessing import LabelEncoder

#fill in mean for floats
for c in train.columns:
    if train[c].dtype=='float16' or  train[c].dtype=='float32' or  train[c].dtype=='float64':
        train[c].fillna(train[c].mean())

#fill in -999 for categoricals
train = train.fillna(-999)
# Label Encoding
for f in train.columns:
    if train[f].dtype=='object': 
        lbl = LabelEncoder()
        lbl.fit(list(train[f].values))
        train[f] = lbl.transform(list(train[f].values))
        
print('Labelling done.')

Labelling done.


In [22]:
#Robin East https://www.kaggle.com/code/robineast/log-loss-example

# Generate data
X, y = make_blobs(n_samples=1000, n_features=2, random_state=42,
                  cluster_std=5.0)
X_train, y_train = X[:600], y[:600]
X_valid, y_valid = X[600:800], y[600:800]
X_train_valid, y_train_valid = X[:800], y[:800]
X_test, y_test = X[800:], y[800:]

# Train uncalibrated random forest classifier on whole train and validation
# data and evaluate on test data
clf = RandomForestClassifier(n_estimators=25)
clf.fit(X_train_valid, y_train_valid)
clf_probs = clf.predict_proba(X_test)
score = log_loss(y_test, clf_probs)
print(score)

1.3174985962299857


#Old age ain't No Place for Sissies

Nice Score!

![](https://encrypted-tbn0.gstatic.com/images?q=tbn:ANd9GcT_H0CCIY0VWwhwaSi-s5M0V9Iq_53cRsYvrw&usqp=CAU)png attitude

#Acknowledgements:

Robin East https://www.kaggle.com/code/robineast/log-loss-example